# ANDASOL-II Variable Thermal Load

> [!info] Definición nueva entrada modelo: factor de carga de la turbina 
> Una variable que vaya de 0 a 1, donde 1 sea la potencia nominal de la turbina y 0 sea que el sistema esté parado. Internamente el modelo debería establecer un valor mínimo de carga parcial por debajo del cual se desactiva la turbina (x<0.5 -> 0)

> [!info] Definición plan de operación diario
> Vector con valores del factor de carga de la turbina para cada muestreo del día. Es decir, si el modelo se evalúa cada hora, en un día sería un vector de 24 valores (1X24): 
> x=[0.5, 0.5, 0.5, 1, 1, ..., 1, 1, 0.5, 0.5]

- Generar 5 planes diarios de operación distintos (o los que veamos apropiados). Que vayan, desde la estrategia actual de generar siempre lo máximo que se pueda en la turbina (plan de operación con todo unos), hasta una alternativa que opere  más horas a carga parcial en las horas centrales.
- En el bucle de simulación, para cada día se evalúan los planes de operación, empezando por el plan que más evite las horas centrales hasta la operación actual. Cuando se encuentre un plan que no fuerce a que haya desenfoque, se usa ese para la simulación. Si no hay ninguno, se coge el último (se produce siempre todo lo que se pueda) pues hay exceso de energía y no hay nada que se pueda hacer.

Pseudo-código de la implementación:

```python
candidatos planes de operación = [
	[0.5,..., 0.5,...,0.5, ..., 1], # Prioriza desplazar generación lo más tarde posible 
	...
	[0.5, ..., 0.5, ..., 1], # Estrategia intermedia
	...
	[1,1,...,1,1,...,1,1] # Prioriza generación instantánea - Estrategia actual
]

# Bucle de simulación
Para cada día en el año:
	Para cada plan candidato en candidatos planes de operación:
		desenfoque = modelo andasol(plan candidato)
		Si no desenfoque:
			# el plan de operación evaluado no desperdicia energía
			FIN día
```

Con 5 planes de operación, en el peor escenario (todos desperdician energía), el tiempo de simulación de un día se vería multiplicado por 5. Viendo los datos, la mayoría de días del año no se operan muchas horas al día, por lo que lo más probable es que en muchos casos los primeros planes ya sean factibles y entonces haya poca penalización en el tiempo de simulación (igual o X2).


In [1]:
import datetime as dt
from pathlib import Path
from loguru import logger

from solhycool_evaluation import OperationPlan, OperationValues as ov, PeriodType
from solhycool_evaluation.utils import generate_annual_operation_table

output_path = Path("../results/operation_plan_andasol")

year=2025
freq='10min'
timezone='Europe/Madrid'

periods: PeriodType = [
    (dt.time(7, 0),   dt.time(9, 30)),
    (dt.time(9, 30),  dt.time(19, 0)),
    (dt.time(19, 0),  dt.time(21, 30)),
    (dt.time(21, 30), dt.time(7, 0)),
]

op_plans: list[OperationPlan] = [
   OperationPlan(periods, [ov.PEAK.value, ov.OFF.value, ov.PEAK.value, ov.PARTIAL.value]),
   OperationPlan(periods, [ov.PEAK.value, ov.OFF.value, ov.PEAK.value, ov.PEAK.value]),
   OperationPlan(periods, [ov.PEAK.value, ov.PARTIAL.value, ov.PEAK.value, ov.PEAK.value]),
   OperationPlan(periods, [ov.PEAK.value, ov.PEAK.value, ov.PEAK.value, ov.PEAK.value]),
]
op_plans

[OperationPlan(period=[(datetime.time(7, 0), datetime.time(9, 30)), (datetime.time(9, 30), datetime.time(19, 0)), (datetime.time(19, 0), datetime.time(21, 30)), (datetime.time(21, 30), datetime.time(7, 0))], values=[1.0, 0.0, 1.0, 0.6]),
 OperationPlan(period=[(datetime.time(7, 0), datetime.time(9, 30)), (datetime.time(9, 30), datetime.time(19, 0)), (datetime.time(19, 0), datetime.time(21, 30)), (datetime.time(21, 30), datetime.time(7, 0))], values=[1.0, 0.0, 1.0, 1.0]),
 OperationPlan(period=[(datetime.time(7, 0), datetime.time(9, 30)), (datetime.time(9, 30), datetime.time(19, 0)), (datetime.time(19, 0), datetime.time(21, 30)), (datetime.time(21, 30), datetime.time(7, 0))], values=[1.0, 0.6, 1.0, 1.0]),
 OperationPlan(period=[(datetime.time(7, 0), datetime.time(9, 30)), (datetime.time(9, 30), datetime.time(19, 0)), (datetime.time(19, 0), datetime.time(21, 30)), (datetime.time(21, 30), datetime.time(7, 0))], values=[1.0, 1.0, 1.0, 1.0])]

In [2]:
# Generate the annual operation table
# You can change the frequency as needed: 'h' (hourly), '30min' (30 min), '15min' (15 min), etc.
annual_table = generate_annual_operation_table(op_plans, year=year, freq=freq, timezone=timezone, extra_cols=False)

# Save to CSV
annual_table.to_csv(output_path / "annual_operation_table_andasol.csv", index_label='timestamp')
logger.info(f"Annual operation table saved to {output_path / 'annual_operation_table_andasol.csv'}")

# Display first few rows
print("\nFirst 24 hours (one day sample):")
annual_table.head(24)

Table shape: (52560, 4)
Date range: 2025-01-01 00:00:00+01:00 to 2025-12-31 23:50:00+01:00
Frequency: <10 * Minutes>

Column names:
['operation_plan_1', 'operation_plan_2', 'operation_plan_3', 'operation_plan_4']


2025-09-03 13:37:40.735 | INFO     | __main__:<module>:7 - Annual operation table saved to ../results/operation_plan_andasol/annual_operation_table_andasol.csv



First 24 hours (one day sample):


,operation_plan_1,operation_plan_2,operation_plan_3,operation_plan_4
2025-01-01 00:00:00+01:00,0.6,1.0,1.0,1.0
2025-01-01 00:10:00+01:00,0.6,1.0,1.0,1.0
2025-01-01 00:20:00+01:00,0.6,1.0,1.0,1.0
2025-01-01 00:30:00+01:00,0.6,1.0,1.0,1.0
2025-01-01 00:40:00+01:00,0.6,1.0,1.0,1.0
2025-01-01 00:50:00+01:00,0.6,1.0,1.0,1.0
2025-01-01 01:00:00+01:00,0.6,1.0,1.0,1.0
2025-01-01 01:10:00+01:00,0.6,1.0,1.0,1.0
2025-01-01 01:20:00+01:00,0.6,1.0,1.0,1.0
2025-01-01 01:30:00+01:00,0.6,1.0,1.0,1.0


In [8]:
# Let's examine the operation patterns for the first few days
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots


annual_table = generate_annual_operation_table(op_plans, year=year, freq=freq, timezone=timezone, extra_cols=True)

# Show sample data for the first day
first_day = annual_table[annual_table['date'] == annual_table['date'].iloc[0]]
# print("Operation values for January 1st, 2025:")
# print(first_day[['operation_plan_1', 'operation_plan_2', 'operation_plan_3', 'operation_plan_4']])

# Create subplots for the operation plans
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f'Operation Plan {i+1}' for i in range(4)],
    vertical_spacing=0.15,
    horizontal_spacing=0.1
)

# Colors for each plan
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
height=600
width=900

# Add traces for each operation plan
for i in range(4):
    row = (i // 2) + 1
    col = (i % 2) + 1
    col_name = f'operation_plan_{i+1}'
    
    fig.add_trace(
        go.Scatter(
            x=first_day.index.hour,
            y=first_day[col_name],
            mode='lines+markers',
            name=f'Plan {i+1}',
            line=dict(color=colors[i], width=3),
            marker=dict(size=6),
            showlegend=False
        ),
        row=row, col=col
    )

# Update layout
fig.update_layout(
    title={
        'text': 'Daily Operation Plans - January 1st, 2025',
        'x': 0.5,
        'font': {'size': 16}
    },
    height=height,
    width=width
)

# Update x and y axes for all subplots
fig.update_xaxes(
    title_text="Hour of Day",
    tickmode='linear',
    tick0=0,
    dtick=2,
    range=[-0.5, 23.5]
)
fig.update_yaxes(
    title_text="Operation Value",
    range=[-0.1, 1.1]
)

fig.show()
fig.write_image(output_path / "daily_operation_plans_andasol.png", width=width, height=height)

# Create an overview plot comparing all operation plans
fig_overview = go.Figure()

operation_cols = [col for col in annual_table.columns if col.startswith('operation_plan')]

for i, col in enumerate(operation_cols):
    fig_overview.add_trace(
        go.Scatter(
            x=first_day.index.hour,
            y=first_day[col],
            mode='lines+markers',
            name=f'Operation Plan {i+1}',
            line=dict(color=colors[i], width=3),
            marker=dict(size=6)
        )
    )

height=500
width=900

fig_overview.update_layout(
    title={
        'text': 'Comparison of All Operation Plans - January 1st, 2025',
        'x': 0.5,
        'font': {'size': 16}
    },
    xaxis_title="Hour of Day",
    yaxis_title="Operation Value",
    xaxis=dict(
        tickmode='linear',
        tick0=0,
        dtick=2,
        range=[-0.5, 23.5]
    ),
    yaxis=dict(range=[-0.1, 1.1]),
    height=height,
    width=width,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

fig_overview.show()
fig_overview.write_image(output_path / "overview_operation_plans_andasol.png", width=width, height=height)

# Summary statistics
# print("\nSummary statistics for operation plans:")
# print(annual_table[operation_cols].describe())

# Create continuous heatmaps showing operation patterns for the whole year

# Use continuous colorscale 'agsunset' for better visual appeal
# Add '_r' at the end to reverse the colorscale
colorscale = 'sunset'

# Create subplots for all operation plans
fig_heatmaps = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f'Operation Plan {i+1}' for i in range(4)],
    vertical_spacing=0.15,
    horizontal_spacing=0.05,
    specs=[[{"type": "heatmap"}, {"type": "heatmap"}],
           [{"type": "heatmap"}, {"type": "heatmap"}]]
)

# Sample data for better visualization (every 7 days to show weekly patterns)
sample_data = annual_table[::7*24]  # Sample every week (7 days * 24 hours)

for plan_idx in range(4):
    row = (plan_idx // 2) + 1
    col = (plan_idx % 2) + 1
    col_name = f'operation_plan_{plan_idx+1}'
    
    # Pivot data for heatmap
    heatmap_data = annual_table.pivot_table(
        index=annual_table.index.hour,
        columns=annual_table.index.dayofyear,
        values=col_name,
        aggfunc='first'
    )
    
    # Add heatmap trace
    fig_heatmaps.add_trace(
        go.Heatmap(
            z=heatmap_data.values,
            x=heatmap_data.columns,
            y=heatmap_data.index,
            colorscale=colorscale,
            zmin=0.0,
            zmax=1.0,
            colorbar=dict(
                title="Operation Value",
                x=1.02 if col == 2 else None,
                len=0.4
            ) if plan_idx == 3 else None,  # Only show colorbar for last plot
            showscale=True if plan_idx == 3 else False
        ),
        row=row, col=col
    )

# Update layout
fig_heatmaps.update_layout(
    title={
        'text': 'Annual Operation Plans - Hourly Patterns (2025)',
        'x': 0.5,
        'font': {'size': 16}
    },
    height=800,
    width=1400
)

# Update x and y axes for all subplots
fig_heatmaps.update_xaxes(title_text="Day of Year")
fig_heatmaps.update_yaxes(title_text="Hour of Day")

fig_heatmaps.show()
fig_heatmaps.write_image(output_path / "annual_operation_plans_heatmaps_andasol.png", width=width, height=height)

# Create a more detailed view for a specific month (January)
january_data = annual_table[annual_table.index.month == 1]

fig_january = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f'Operation Plan {i+1} - January 2025' for i in range(4)],
    vertical_spacing=0.15,
    horizontal_spacing=0.05
)

for plan_idx in range(4):
    row = (plan_idx // 2) + 1
    col = (plan_idx % 2) + 1
    col_name = f'operation_plan_{plan_idx+1}'
    
    # Pivot data for January heatmap
    january_heatmap = january_data.pivot_table(
        index=january_data.index.hour,
        columns=january_data.index.day,
        values=col_name,
        aggfunc='first'
    )
    
    fig_january.add_trace(
        go.Heatmap(
            z=january_heatmap.values,
            x=january_heatmap.columns,
            y=january_heatmap.index,
            colorscale=colorscale,
            zmin=0.0,
            zmax=1.0,
            colorbar=dict(
                title="Operation Value",
                x=1.02 if col == 2 else None,
                len=0.4
            ) if plan_idx == 3 else None,
            showscale=True if plan_idx == 3 else False
        ),
        row=row, col=col
    )

fig_january.update_layout(
    title={
        'text': 'January 2025 - Daily Operation Plans',
        'x': 0.5,
        'font': {'size': 16}
    },
    height=800,
    width=1200
)

fig_january.update_xaxes(title_text="Day of Month")
fig_january.update_yaxes(title_text="Hour of Day")

fig_january.show()
fig_january.write_image(output_path / "january_operation_plans_heatmaps_andasol.png", width=width, height=height)

Table shape: (52560, 8)
Date range: 2025-01-01 00:00:00+01:00 to 2025-12-31 23:50:00+01:00
Frequency: <10 * Minutes>

Column names:
['operation_plan_1', 'operation_plan_2', 'operation_plan_3', 'operation_plan_4', 'date', 'time', 'day_of_year', 'weekday']


In [5]:
str(output_path / "january_operation_plans_heatmaps_andasol.png")

'../results/operation_plan_andasol/january_operation_plans_heatmaps_andasol.png'